In [1]:
from pathlib import Path
import shutil
import csv
import numpy as np
import pandas as pd
import json

In [8]:
path_full_img = Path('../final_dataset/test/images')
path_full_labels = Path('../final_dataset/test/labels')
path_test_img = Path('../lite_dataset/test/images')
path_test_labels = Path('../lite_dataset/test/labels')

In [11]:
if not path_test_img.exists() and not path_test_labels.exists():
    try:
        path_test_img.mkdir(parents=True)
        path_test_labels.mkdir()
        print('Path test dataset created!')
    except Exception as e:
        print('Path test dataset creation failed!' , e)

Path test dataset created!


In [12]:
# copy img and labels file to test folder
# where TEST_NUM is number of files
TEST_NUM = 200
itr = 1
for file in path_full_img.glob('*.jpg'):
    print(itr,'   ',file)
    label_path= Path(path_full_labels,file.stem+'.txt')
    shutil.copy(file, path_test_img)
    shutil.copy(label_path, path_test_labels)
    print(label_path)
    
    
    if itr >= TEST_NUM:
        break
    itr += 1

1     ..\final_dataset\test\images\000106393cfe2343888c584e65fd2274.jpg
..\final_dataset\test\labels\000106393cfe2343888c584e65fd2274.txt
2     ..\final_dataset\test\images\0039eb1ff33d29c55f943e05730bb259.jpg
..\final_dataset\test\labels\0039eb1ff33d29c55f943e05730bb259.txt
3     ..\final_dataset\test\images\0041e69431bf872309d1aff628b6494f.jpg
..\final_dataset\test\labels\0041e69431bf872309d1aff628b6494f.txt
4     ..\final_dataset\test\images\0049943dd024c06611a90feb5acebbbd.jpg
..\final_dataset\test\labels\0049943dd024c06611a90feb5acebbbd.txt
5     ..\final_dataset\test\images\009a5b410ff34d171280c1135e15c011.jpg
..\final_dataset\test\labels\009a5b410ff34d171280c1135e15c011.txt
6     ..\final_dataset\test\images\00b627cb5578bb036edb01cdcf7b56d9.jpg
..\final_dataset\test\labels\00b627cb5578bb036edb01cdcf7b56d9.txt
7     ..\final_dataset\test\images\011ced9d123e0da9c289e94a3b11165e.jpg
..\final_dataset\test\labels\011ced9d123e0da9c289e94a3b11165e.txt
8     ..\final_dataset\test\images

In [13]:
#creates csv for labels in test

with (open(path_test_img/'metadata.csv', 'w', newline='') as output_file):
    csv_writer = csv.writer(output_file)
    csv_writer.writerow(['file_name','label','x_center', 'y_center', 'width', 'height'])
    
    for file in path_test_labels.glob('*.txt'):
        with open(file,'r') as f:
            lines = f.readlines()
            file_name = file.stem+'.jpg'
            for line in lines:
                row = [file_name]
                row.extend(line.split(' '))
                row[-1]=row[-1].strip('\n')
                csv_writer.writerow(row)
                print(row)
        f.close()        
    output_file.close()

['000106393cfe2343888c584e65fd2274.jpg', 'F16', '0.2552556818181818', '0.28878281622911695', '0.5099431818181818', '0.5767700875099443']
['0039eb1ff33d29c55f943e05730bb259.jpg', 'C2', '0.4683333333333333', '0.49333333333333335', '0.6655555555555556', '0.3466666666666667']
['0041e69431bf872309d1aff628b6494f.jpg', 'E2', '0.5531665363565286', '0.6186206896551724', '0.7670054730258014', '0.46206896551724136']
['0049943dd024c06611a90feb5acebbbd.jpg', 'Mig31', '0.5930094786729858', '0.45066666666666666', '0.07582938388625593', '0.030222222222222223']
['0049943dd024c06611a90feb5acebbbd.jpg', 'F35', '0.5062203791469194', '0.7528888888888889', '0.06575829383886256', '0.02311111111111111']
['009a5b410ff34d171280c1135e15c011.jpg', 'C5', '0.479296875', '0.5604460093896714', '0.20546875', '0.15140845070422534']
['00b627cb5578bb036edb01cdcf7b56d9.jpg', 'F35', '0.6957692307692308', '0.422722029988466', '0.4576923076923077', '0.3748558246828143']
['011ced9d123e0da9c289e94a3b11165e.jpg', 'V22', '0.5137

In [14]:
# convert labels from csv to json
df = pd.read_csv(path_test_img/'metadata.csv')
df.to_json(path_test_img/'metadata.json', orient='records')

In [16]:
df

,file_name,label,x_center,y_center,width,height
0,000106393cfe2343888c584e65fd2274.jpg,F16,0.255256,0.288783,0.509943,0.576770
1,0039eb1ff33d29c55f943e05730bb259.jpg,C2,0.468333,0.493333,0.665556,0.346667
2,0041e69431bf872309d1aff628b6494f.jpg,E2,0.553167,0.618621,0.767005,0.462069
3,0049943dd024c06611a90feb5acebbbd.jpg,Mig31,0.593009,0.450667,0.075829,0.030222
4,0049943dd024c06611a90feb5acebbbd.jpg,F35,0.506220,0.752889,0.065758,0.023111
...,...,...,...,...,...,...
323,126a0390052f8ffe741733de6763a740.jpg,F35,0.308333,0.599184,0.125556,0.063976
324,126a0390052f8ffe741733de6763a740.jpg,F35,0.184444,0.603693,0.120000,0.065264
325,1272cdf4e852b5e89adbbb3c65cd68ad.jpg,F117,0.698452,0.291040,0.118989,0.070956
326,1272cdf4e852b5e89adbbb3c65cd68ad.jpg,F117,0.840668,0.359892,0.113284,0.064342


# Curating Train dataset

In [6]:
from pathlib import Path
import shutil
from tqdm.notebook import tqdm
import csv
import numpy as np
import pandas as pd
import json

In [5]:
path_full_img = Path('../final_dataset/train/images')
path_full_labels = Path('../final_dataset/train/labels')
path_train_img = Path('../lite_dataset/train/images')
path_train_labels = Path('../lite_dataset/train/labels')

In [4]:
if not path_train_img.exists() and not path_train_labels.exists():
    try:
        path_train_img.mkdir(parents=True)
        path_train_labels.mkdir(parents=True)
        print('Path train dataset created!')
    except Exception as e:
        print('Path train dataset creation failed!' , e)

Path test dataset created!


In [8]:
# copy img and labels file to test folder
# where TEST_NUM is number of files
TRAIN_NUM = 1000
itr = 1
for file in tqdm(path_full_img.glob('*.jpg')):
    # print(itr,'   ',file)
    label_path= Path(path_full_labels,file.stem+'.txt')
    shutil.copy(file, path_train_img)
    shutil.copy(label_path, path_train_labels)
    # print(label_path)
    
    
    if itr >= TRAIN_NUM:
        break
    itr += 1

0it [00:00, ?it/s]

In [9]:
#creates metadata for labels in train

with (open(path_train_img/'metadata.csv', 'w', newline='') as output_file):
    csv_writer = csv.writer(output_file)
    csv_writer.writerow(['file_name','label','x_center', 'y_center', 'width', 'height'])
    
    for file in tqdm(path_train_labels.glob('*.txt')):
        with open(file,'r') as f:
            lines = f.readlines()
            file_name = file.stem+'.jpg'
            for line in lines:
                row = [file_name]
                row.extend(line.split(' '))
                row[-1]=row[-1].strip('\n')
                csv_writer.writerow(row)
                print(row)
        f.close()        
    output_file.close()

0it [00:00, ?it/s]

['00032844ab679240fc03ecd27d29a6aa.jpg', 'F18', '0.516725352112676', '0.3884976525821596', '0.9665492957746479', '0.7582159624413145']
['0003f56298fa8999168d7988a2e9549d.jpg', 'F22', '0.46502718734552645', '0.39451242120875046', '0.8247652001977261', '0.41453466814979606']
['0003f56298fa8999168d7988a2e9549d.jpg', 'F22', '0.5568462679189323', '0.44123099740452354', '0.8863074641621355', '0.4649610678531702']
['000aa01b25574f28b654718db0700f72.jpg', 'F35', '0.69580078125', '0.2490842490842491', '0.5595703125', '0.23882783882783884']
['000aa01b25574f28b654718db0700f72.jpg', 'JAS39', '0.17529296875', '0.6087912087912087', '0.185546875', '0.09084249084249084']
['000aa01b25574f28b654718db0700f72.jpg', 'JAS39', '0.137939453125', '0.7021978021978021', '0.15380859375', '0.073992673992674']
['000aa01b25574f28b654718db0700f72.jpg', 'B52', '0.382080078125', '0.7611721611721611', '0.49365234375', '0.2021978021978022']
['000e7662268a1071827c5a8663e773f9.jpg', 'US2', '0.7175', '0.4304906542056075', '

In [10]:
# convert labels from csv to json
df = pd.read_csv(path_train_img/'metadata.csv')
df.to_json(path_train_img/'metadata.json', orient='records')

In [11]:
df.head()

,file_name,label,x_center,y_center,width,height
0,00032844ab679240fc03ecd27d29a6aa.jpg,F18,0.516725,0.388498,0.966549,0.758216
1,0003f56298fa8999168d7988a2e9549d.jpg,F22,0.465027,0.394512,0.824765,0.414535
2,0003f56298fa8999168d7988a2e9549d.jpg,F22,0.556846,0.441231,0.886307,0.464961
3,000aa01b25574f28b654718db0700f72.jpg,F35,0.695801,0.249084,0.559570,0.238828
4,000aa01b25574f28b654718db0700f72.jpg,JAS39,0.175293,0.608791,0.185547,0.090842


In [13]:
df.corr(numeric_only=True)

,x_center,y_center,width,height
x_center,1.000000,-0.036734,0.040943,0.020795
y_center,-0.036734,1.000000,0.067405,0.082335
width,0.040943,0.067405,1.000000,0.794870
height,0.020795,0.082335,0.794870,1.000000
